In [1]:
import kagglehub

In [2]:
path = kagglehub.dataset_download('jangedoo/utkface-new', path='./')

Using Colab cache for faster access to the 'utkface-new' dataset.


In [3]:
print(f'Dataset downloaded to: {path}')

Dataset downloaded to: /kaggle/input/utkface-new/./


In [5]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [7]:
import os

folder_path = os.path.join(path, 'UTKFace')

In [8]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  # Handle potential non-image files if any remain after fixing the path
  if file.endswith('.jpg'): # Ensure we only process image files
    age.append(int(file.split('_')[0]))
    gender.append(int(file.split('_')[1]))
    img_path.append(file)

In [9]:
len(age)

23708

In [10]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [11]:
df.shape

(23708, 3)

In [12]:
df.head()

,age,gender,img
0,26,0,26_0_2_20170104023102422.jpg.chip.jpg
1,22,1,22_1_1_20170112233644761.jpg.chip.jpg
2,21,1,21_1_3_20170105003215901.jpg.chip.jpg
3,28,0,28_0_0_20170117180555824.jpg.chip.jpg
4,17,1,17_1_4_20170103222931966.jpg.chip.jpg


In [18]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [19]:
train_df.shape

(20000, 3)

In [20]:
test_df.shape

(3708, 3)

In [21]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [23]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [24]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [ ]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [26]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [27]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [28]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [30]:
def create_multi_output_generator(generator):
    for batch_x, batch_y in generator:
        # batch_y is a list: [age_labels_batch, gender_labels_batch]
        # We need to convert it to a dictionary matching the model's output names
        yield batch_x, {'age': batch_y[0], 'gender': batch_y[1]}

In [31]:
model.fit(create_multi_output_generator(train_generator), epochs=10, validation_data=create_multi_output_generator(test_generator))

Epoch 1/10
    571/Unknown 295s 490ms/step - age_loss: 15.9870 - age_mae: 15.9870 - gender_accuracy: 0.5086 - gender_loss: 1.5172 - loss: 166.1878

KeyboardInterrupt: 